# What the model relies on

The chosen model (decision 13): TF-IDF on 1-3 word n-grams + logistic regression. For a linear model the explanation is exact: a review's log-odds of "positive" are the intercept plus, for every term in it, the term's TF-IDF weight times its coefficient (`reviews.explain`, tested to add up to the model's own score). Validation set only.

In [1]:
import json
import os
import re
from pathlib import Path

os.environ.setdefault("REVIEWS_DATA_DIR", "../data")

import numpy as np

from reviews import explain
from reviews.compare import load
from reviews.model_store import load_model
from reviews.scrub import excerpt

grams = json.loads(Path("../reports/ngrams.json").read_text())
model = load_model(Path("../models/ngrams-best"), grams["winner_sha256"]).model
val_x, val_y = load("validation")
y = val_y.to_numpy()
prob = model.predict_proba(val_x)[:, 1]
pred = (prob >= 0.5).astype(int)
round((pred == y).mean(), 4)

np.float64(0.9348)

## The strongest terms for each class

In [2]:
explain.top_terms(model, 25)

,negative term,negative weight,positive term,positive weight
0,not,-29.67,great,35.07
1,disappointing,-25.46,excellent,23.66
2,boring,-23.21,awesome,18.84
3,disappointment,-22.45,good,18.19
4,poor,-22.04,perfect,17.34
5,horrible,-21.33,best,17.06
6,not worth,-20.99,amazing,16.20
7,terrible,-20.61,fantastic,14.61
8,worst,-19.86,not bad,14.10
9,disappointed,-19.71,wonderful,13.95


- Mostly what you'd hope: "disappointing", "boring", "poor" against "great", "excellent", "perfect".
- **"not" is the single strongest negative term**, and the pairs fix its exceptions: "not bad" is strongly positive (+14), "not worth" and "not very" negative.
- **Context the trigrams pick up:** "be disappointed" is *positive*, from "you won't be disappointed"; "my only" is positive, from "my only complaint is..." in an otherwise happy review; "at best" is negative.

## Shortcuts? Star ratings written in the text

"2 stars" and "4 stars" are among the strongest terms. Reviewers sometimes write their rating, and the label comes from the rating, so this could be a shortcut that inflates accuracy.

In [3]:
explain.terms_matching(model, r"\b(?:star|stars)\b", 12)

,term,weight
78,2 stars,-14.808252
2317,two stars,-12.695388
139,4 stars,12.513727
638,four stars,10.229632
215,5 stars,8.804971
1010,one star,-7.427408
22,1 star,-6.319557
592,five stars,5.115991
808,it 4 stars,4.792465
810,it 5 stars,4.451844


In [4]:
STAR = re.compile(r"\b(?:[0-5]|zero|one|two|three|four|five)[ -]stars?\b", re.I)
mentions = np.array([bool(STAR.search(t)) for t in val_x])
{
    "reviews mentioning stars": int(mentions.sum()),
    "share": round(mentions.mean(), 4),
    "accuracy on them": round((pred[mentions] == y[mentions]).mean(), 4),
    "accuracy on the rest": round((pred[~mentions] == y[~mentions]).mean(), 4),
}

{'reviews mentioning stars': 777,
 'share': np.float64(0.0388),
 'accuracy on them': np.float64(0.9241),
 'accuracy on the rest': np.float64(0.9352)}

It isn't a shortcut that matters: only 3.9% of reviews mention stars, and the model is *less* accurate on them (92.4% vs 93.5%). Those reviews are often mixed ("I'd give it 4 stars, but..."). The terms stay in; they're the reviewer's own words.

Odd detail: "2 stars" (-14.8) weighs more than "1 star" (-6.3). A 1-star reviewer rarely bothers to write the number; "2 stars" more often comes with an explanation.

## Product types: a small prior, not a sentiment

In [5]:
explain.terms_matching(
    model, r"^(?:book|dvd|cd|movie|album|kindle|product|item|game|music|film)$", 12
)

,term,weight
9,music,2.389226
0,album,2.385550
2,cd,2.158072
6,item,-1.930613
3,dvd,1.451023
7,kindle,-1.202487
5,game,0.926781
10,product,-0.884689
1,book,0.710417
4,film,0.359640


Music words ("music", "album", "cd") lean positive (+2.2 to +2.4), "item" and "product" negative (-0.9 to -1.9). These are base rates: music reviews in this data are more often positive, and people say "this item" when complaining about a gadget. Next to the sentiment words (15 to 35) they're small, but they mean a neutral review of a CD gets a slight positive nudge.

In [6]:
explain.terms_matching(model, r"\b(?:return|returned|refund)\b", 8)

,term,weight
1261,returned,-9.189018
725,refund,-8.472128
890,return,-6.913832
18,a refund,-4.696870
1313,returned it,-4.630784
465,i returned,-4.529403
1579,to return,-4.397388
1216,return to,3.138618


Returns and refunds are strongly negative, a first hint of what the complaint topics (day 8) will find.

## One review at a time

In [7]:
def show(i, n=8):
    print(f"true {'positive' if y[i] else 'negative'}, predicted {prob[i]:.2f} positive")
    print(excerpt(val_x[i], 260))
    return explain.contributions(model, val_x[i], n).round(2)


# A correct prediction that needed a pair: contains 'not bad'.
not_bad = [i for i, t in enumerate(val_x) if "not bad" in t.lower() and pred[i] == y[i] == 1]
show(not_bad[0])

true positive, predicted 1.00 positive
Not bad at all! I've listened to the CD about 4 times now and I can honestly say it's growing on me. Like I Love You is a great song and is worth buying the album for, but he's put together an album that has a lot of variety and is a lot of fun. My personal…


,term,contribution
0,not bad,0.71
1,great,0.65
2,bad,-0.42
3,not,-0.41
4,fun,0.37
5,favorites,0.31
6,love,0.30
7,at all,-0.27


In [8]:
# The most confident mistake: a review the model was sure about and got wrong.
wrong = np.where(pred != y)[0]
worst = wrong[np.argmax(np.abs(prob[wrong] - 0.5))]
show(worst)

true positive, predicted 0.00 positive
Okay... This is only an okay book which I can download it in [url]. Why would I waste my money on this P.C.? (Piece of Crap)


,term,contribution
0,crap,-1.26
1,okay,-1.11
2,waste,-0.99
3,my money,-0.62
4,money,-0.56
5,an okay,-0.45
6,of crap,-0.43
7,why would,-0.41


The most confident "mistake" is the label's: the text calls the book a waste of money and "P.C. (Piece of Crap)", and the star rating says positive. The model is right about the text. Day 7 counts how many mistakes are like this.

In [9]:
# A mistake close to the line.
close = wrong[np.argmin(np.abs(prob[wrong] - 0.5))]
show(close)

true positive, predicted 0.50 positive
A strange book but fascinating none the less... Dillard weaves a web of mystery, both of spiritual and ecological things. She does not hide her deep knowledge of the "other realm" yet she does not preach "earth worship". Her writing style is definetly…


,term,contribution
0,not,-1.15
1,fascinating,0.63
2,does not,-0.49
3,stick with,-0.39
4,yet,0.29
5,stick,-0.28
6,the less,0.25
7,a strange,-0.21


## Takeaways

1. The model leans on sentiment words and short phrases, not on topic or product type.
2. Pairs and triples handle the common negations ("not bad", "not worth") and some context ("won't be disappointed", "my only complaint").
3. Star ratings in the text are a strong signal but rare (3.9%) and don't inflate accuracy.
4. Product-type words carry small base-rate effects (music positive, "item" negative).
5. Confident mistakes are worth reading one by one: that's day 7.